In [ ]:
"""
SVM grid search on Topological Representations. Example: Betti Curves.

Output:
    - CSV with CV performance for all configurations
"""

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, make_scorer, fbeta_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# Configuration
BASE = "RIPS"

DIMENSIONS = ["H0", "H1", "H0H1"]
FOLDER_TEMPLATE = "BettiCurves/{}"

CV_SPLITS = 6
RANDOM_STATE = 0

OVERSAMPLING = [
    ("no_ros", False),
    ("ros", True),
]

OUT_CSV = "svm_grid.csv"


SVM_GRID = [
    {"kernel": "linear", "C": [0.1, 10]},
    {"kernel": "rbf", "C": [0.1, 10], "gamma": [0.01, 0.1]},
    {"kernel": "poly", "C": [0.1, 10], "gamma": [0.01, 0.1], "degree": [2, 3], "coef0": [0, 1]},
    {"kernel": "sigmoid", "C": [0.1, 10], "gamma": [0.01, 0.1], "coef0": [0, 1]},
]

# LOAD DATA
def load_features(folder: Path):
    files = sorted([f for f in folder.iterdir() if f.is_file() and not f.name.startswith(".")])
    data = []
    for f in files:
        try:
            v = np.loadtxt(f).flatten()
            data.append(v)
        except:
            continue
    return data


def expand_grid(block):
    keys = list(block.keys())
    values = [v if isinstance(v, list) else [v] for v in block.values()]

    for combo in product(*values):
        params = dict(zip(keys, combo))

        # ensure numeric types
        if "C" in params:
            params["C"] = float(params["C"])
        if "gamma" in params and params["gamma"] is not None:
            params["gamma"] = float(params["gamma"])
        if "coef0" in params and params["coef0"] is not None:
            params["coef0"] = float(params["coef0"])
        if "degree" in params and params["degree"] is not None:
            params["degree"] = int(params["degree"])

        yield params


def iter_svm_params(grid):
    for block in grid:
        yield from expand_grid(block)


def build_pipeline(use_ros, params):

    steps = [("scaler", StandardScaler())]

    if use_ros:
        steps.append(("ros", RandomOverSampler(random_state=RANDOM_STATE)))

    steps.append((
        "svm",
        SVC(
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            **params
        )
    ))

    return ImbPipeline(steps)

f2 = make_scorer(fbeta_score, beta=2, zero_division=0)

scoring = {
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
    "f2": f2
}

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)



# Main
rows = []

for dim in DIMENSIONS:

    dirR = Path(BASE) / FOLDER_TEMPLATE.format(dim) / "Relapse"
    dirNR = Path(BASE) / FOLDER_TEMPLATE.format(dim) / "NonRelapse"

    X_R = load_features(dirR)
    X_NR = load_features(dirNR)

    X = np.vstack(X_NR + X_R)
    y = np.array([0]*len(X_NR) + [1]*len(X_R))

    for name, use_ros in OVERSAMPLING:
        for params in iter_svm_params(SVM_GRID):

            clf = build_pipeline(use_ros, params)

            scores = cross_validate(
                clf, X, y,
                cv=cv,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False
            )

            preds = cross_val_predict(clf, X, y, cv=cv, n_jobs=-1)
            cm = confusion_matrix(y, preds)

            rows.append({
                "model": "SVM",
                "dimension": dim,
                "oversampling": name,
                "auc": np.mean(scores["test_roc_auc"]),
                "accuracy": np.mean(scores["test_accuracy"]),
                "f2": np.mean(scores["test_f2"]),
                "cm": f"({cm[0,0]} {cm[0,1]}; {cm[1,0]} {cm[1,1]})",
                "kernel": params.get("kernel"),
                "C": params.get("C"),
                "gamma": params.get("gamma"),
                "degree": params.get("degree"),
                "coef0": params.get("coef0"),
            })

    print(f"Done: {dim}")


df = pd.DataFrame(rows).sort_values(
    ["dimension", "oversampling", "f2"],
    ascending=[True, True, False]
).reset_index(drop=True)

df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)


# Best config
best = df.groupby(["dimension", "oversampling"], as_index=False).head(1)

print("\nBest configs:")
print(best[["dimension", "oversampling", "auc", "accuracy", "f2"]])